# Buổi 7: Lần hoàn thành đầu và transaction
**GIẢ LẬP**
## Câu hỏi → dự đoán
Mỗi dòng kết quả sẽ là người, phiên, lượt thử hay nhóm? Hãy viết câu trả lời trước khi chạy.

In [ ]:
from pathlib import Path
import sys
root=Path.cwd()
while not (root/'game_lab').is_dir() and root!=root.parent: root=root.parent
if not (root/'game_lab').is_dir(): raise RuntimeError('Mở bằng python run_game.py --notebook tại thư mục dự án')
sys.path.insert(0,str(root))
from game_lab.learning import read_sql,learning_db
db=learning_db()
print('Database chỉ đọc:',db)


## SQL mẫu và giải thích
Đánh số tất cả lượt trước khi tìm lần completed đầu. Không chỉ đánh số lượt completed.

In [ ]:
result=read_sql("WITH ranked AS (SELECT s.player_id,a.level_id,a.result,ROW_NUMBER() OVER(PARTITION BY s.player_id,a.level_id ORDER BY a.started_at,a.attempt_id) AS attempt_number FROM level_attempts a JOIN sessions s USING(session_id)) SELECT player_id,level_id,MIN(CASE WHEN result='completed' THEN attempt_number END) AS attempts_to_first_complete FROM ranked GROUP BY player_id,level_id LIMIT 20;")
result

## Thay đổi để quan sát
Dự đoán số dòng, chạy truy vấn thứ hai rồi giải thích điều kiện JOIN/lọc/nhóm.

In [ ]:
read_sql("SELECT l.name,COUNT(DISTINCT CASE WHEN a.result='completed' THEN s.player_id END) AS completed_players FROM levels l LEFT JOIN level_attempts a USING(level_id) LEFT JOIN sessions s USING(session_id) GROUP BY l.level_id ORDER BY l.level_order;")

## Rollback trên SQLite tạm
Không ghi DB nguồn. Tạo lỗi khóa chính và kiểm tra toàn thao tác bị hủy.

In [ ]:
import sqlite3
c=sqlite3.connect(':memory:')
c.execute('CREATE TABLE demo(id INTEGER PRIMARY KEY)')
try:
    with c:
        c.execute('INSERT INTO demo VALUES(1)')
        c.execute('INSERT INTO demo VALUES(1)')
except sqlite3.IntegrityError: pass
assert c.execute('SELECT COUNT(*) FROM demo').fetchone()[0]==0
c.close()

## Tự làm và kiểm tra
Làm các bài tương ứng trong docs/exercises.md trước khi xem solutions.md. Kiểm tra số dòng, mẫu số, NULL và phạm vi thời gian.

In [ ]:
assert int(read_sql('SELECT COUNT(*) AS n FROM sessions').iloc[0,0]) >= int(read_sql('SELECT COUNT(DISTINCT player_id) AS n FROM sessions').iloc[0,0])
print('Số phiên không nhỏ hơn số người có phiên')